# FORGE-MA: TRL Fine-Tuning with GRPO
This notebook fine-tunes a local LLM (Qwen 0.5B) using GRPO to maximize Tactic Edit Distance (TED) reward against the FORGE-MA multi-agent environment.

In [ ]:
# Cell 1 — Install & Universal Environment Setup (Colab / Kaggle / Local)
import os, sys, subprocess
import torch

HOME_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else ("/content" if os.path.exists("/content") else os.getcwd())
REPO_DIR = os.path.join(HOME_DIR, "forge-rl")
UNSLOTH_PKG = "unsloth[kaggle-new]" if "kaggle" in HOME_DIR else "unsloth[colab-new]"

print(f"Setting up environment for {HOME_DIR}...")

# Run installations
cmds = [
    "pip install -q trl transformers mergekit datasets accelerate peft bitsandbytes",
    "pip install -q \"openenv-core[core]>=0.2.1\"",
    f"pip install -q \"{UNSLOTH_PKG} @ git+https://github.com/unslothai/unsloth.git\"",
    "pip install -q torch-geometric",
    f"rm -rf {REPO_DIR}",
    f"git clone https://github.com/Godhand-Arnav/Scalar-finals.git {REPO_DIR}"
]

for cmd in cmds:
    print(f"Running: {cmd}")
    os.system(cmd)

# Set working directory safely
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

if not torch.cuda.is_available():
    print('WARNING: NO GPU DETECTED. Please enable GPU in your notebook settings.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete.')

In [ ]:
import sys, os
!git clone https://huggingface.co/spaces/NeuralHU/forge-rl /content/forge-rl
sys.path.insert(0, "/content/forge-rl")
os.chdir("/content/forge-rl")
print("Setup complete")

In [ ]:
from rewards.tactic_edit_dist import tactic_edit_distance
from env.primitives import PrimitiveType
import re, json

def extract_chain(text: str) -> list:
    """Parse primitive chain from model output."""
    primitive_names = [p.value for p in PrimitiveType]
    found = []
    for name in primitive_names:
        if name in text.upper():
            found.append(name)
    return found[:4]

def reward_fn(completions, prompts=None, true_chains=None, **kwargs):
    """
    GRPO reward function.
    Scores each completion by TED against the true chain.
    """
    rewards = []
    for i, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        predicted = extract_chain(text)
        true = true_chains[i] if true_chains and i < len(true_chains) else []
        ted = tactic_edit_distance(predicted, true)
        rewards.append(float(ted))
    return rewards

print("Reward function defined. Test:")
test_ted = tactic_edit_distance(["SOURCE_LAUNDER", "TEMPORAL_SHIFT"], ["SOURCE_LAUNDER", "TEMPORAL_SHIFT", "QUOTE_FABRICATE"])
print(f"  TED test: {test_ted:.3f} (expected ~0.70)")

In [ ]:
from env.forge_env import ForgeEnv
import json

env = ForgeEnv()
training_data = []
print("Collecting 50 training episodes...")

for ep in range(50):
    obs, info = env.reset()
    true_chain = info.get("true_chain", [])
    claim = info.get("claim_text", "A claim about health research.")
    
    training_data.append({
        "prompt": f"""You are a forensic analyst. Given this claim, identify which deception 
primitives were used to fabricate it. Choose from: SOURCE_LAUNDER, TEMPORAL_SHIFT, 
ENTITY_SUBSTITUTE, QUOTE_FABRICATE, CONTEXT_STRIP, CITATION_FORGE, NETWORK_AMPLIFY, SATIRE_REFRAME.

Claim: {claim}

List the primitives used, in order:""",
        "true_chain": [str(p.value) if hasattr(p, 'value') else str(p) for p in true_chain],
    })
    if (ep + 1) % 10 == 0:
        print(f"  Collected {ep + 1}/50 episodes")

print(f"Training data ready: {len(training_data)} episodes")

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import inspect

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # fits in T4 memory

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Version-safe config builder
_valid = set(inspect.signature(GRPOConfig).parameters.keys())
_cfg = dict(
    output_dir="./forge_ma_grpo",
    num_train_epochs=1,
    max_steps=100,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=50,
    report_to="none",
    fp16=True,
    num_generations=4,
    generation_batch_size=4,
)

# max_completion_length (TRL 1.x) or max_new_tokens (TRL 0.x)
for k, v in [('max_completion_length', 64), ('max_new_tokens', 64)]:
    if k in _valid:
        _cfg[k] = v
        break

config = GRPOConfig(**_cfg)

# Build dataset
from datasets import Dataset
def build_dataset(data):
    return Dataset.from_list([
        {"prompt": d["prompt"], "true_chain": d["true_chain"]}
        for d in data
    ])

dataset = build_dataset(training_data)

def reward_wrapper(completions, prompts=None, **kwargs):
    true_chains = [training_data[i % len(training_data)]["true_chain"]
                   for i in range(len(completions))]
    return reward_fn(completions, prompts, true_chains)

try:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_wrapper,
        args=config,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=reward_wrapper,
        args=config,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )

print("Starting GRPO training (100 steps)...")
trainer.train()
print("Training complete!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as mplstyle
import os
mplstyle.use('dark_background')
os.makedirs("baselines", exist_ok=True)

steps, rewards = [], []
REWARD_LOG_KEYS = (
    'rewards/mean', 'reward/mean', 'reward',
    'train/reward', 'rewards/reward_mean', 'mean_reward', 'train_reward'
)

if 'trainer' in locals():
    history = trainer.state.log_history
    for h in history:
        if 'step' not in h: continue
        for key in REWARD_LOG_KEYS:
            if key in h:
                steps.append(h['step'])
                rewards.append(h[key])
                break
else:
    print("Trainer not found! Using dummy data to test plot...")
    steps, rewards = [10, 20, 30], [0.1, 0.5, 0.8]

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, color="#22d3ee", linewidth=2, label="FORGE-MA TED reward")
plt.axhline(y=0.11, color="#f87171", linestyle="--", alpha=0.6, label="Random baseline")
plt.xlabel("Training step", color="#94a3b8")
plt.ylabel("Mean TED reward", color="#94a3b8")
plt.title("FORGE-MA GRPO Training Curve", color="#f8fafc", pad=15)
plt.legend()
plt.tight_layout()
plt.savefig("baselines/grpo_reward_curve.png", dpi=150, bbox_inches="tight",
            facecolor="#080a14")
plt.show()
print("Reward curve saved to baselines/grpo_reward_curve.png")

In [ ]:
def evaluate_agent(model, tokenizer, episodes=20, label=""):
    teds = []
    env = ForgeEnv()
    for ep in range(episodes):
        obs, info = env.reset()
        true_chain = info.get("true_chain", [])
        claim = info.get("claim_text", "A health claim.")
        prompt = training_data[0]["prompt"].replace(
            training_data[0]["prompt"].split("Claim:")[-1],
            f"\n{claim}\n\nList the primitives used, in order:"
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        text = tokenizer.decode(out[0], skip_special_tokens=True)
        predicted = extract_chain(text)
        true = [str(p.value) if hasattr(p,'value') else str(p) for p in true_chain]
        teds.append(tactic_edit_distance(predicted, true))
    mean = sum(teds)/len(teds)
    print(f"  {label}: mean TED = {mean:.3f}")
    return mean

print("Evaluating trained model...")
trained_ted = evaluate_agent(model, tokenizer, episodes=20, label="After GRPO (100 steps)")
print(f"\nImprovement: random(0.11) → v0(~0.14) → after training({trained_ted:.3f})")